# AnnData Copilot: LLM-Assisted Spatial Biology Analytics

**Author:** Mohamed Masoud

This section adds an **AnnData Copilot** layer on top of the real `adata_hvg` object generated in this workflow.

The functions below query the processed Visium-derived AnnData object and the marker tables already computed above.

Our goal of this section  is to demonstrate a practical AI-ready interface for:

- cluster-level marker gene questions
- gene expression summaries by Leiden cluster
- spatial location summaries
- spatial autocorrelation / Moran's I summaries
- LLM-assisted biological interpretation grounded in computed outputs

Parent Project: SpatialBio https://github.com/Mmasoud1/SpatialBio/tree/main

> **Note:**  This is an exploratory research software demo, not a clinical or diagnostic decision-making tool.


## Environment setup


In [ ]:
!pip -q install -U scanpy anndata pandas numpy openai


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 9.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 156.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 165.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.


## Clone spatialBio repo

In [ ]:
!git clone https://github.com/Mmasoud1/SpatialBio.git
%cd SpatialBio

Cloning into 'SpatialBio'...
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 29 (delta 0), reused 5 (delta 0), pack-reused 23 (from 1)
Receiving objects: 100% (29/29), 90.85 MiB | 14.52 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/SpatialBio


## Load saved workflow objects

This notebook expects `adata_hvg.h5ad` and `cluster_markers.csv` files generated by `SpatialBio_Transcriptomics_workflow.ipynb` and saved under `data/processed`.


In [ ]:
from pathlib import Path

ROOT = Path.cwd()
ADATA_PATH = ROOT / "data" / "processed" / "adata_hvg.h5ad"
MARKERS_PATH = ROOT / "data" / "processed" / "cluster_markers.csv"

In [ ]:
import scanpy as sc
import pandas as pd

adata_path = Path(ADATA_PATH)
markers_path = Path(MARKERS_PATH)

if not adata_path.exists() or not markers_path.exists():
    raise FileNotFoundError(
        f"Missing copilot data files.\n"
        f"Expected:\n"
        f"  {adata_path}\n"
        f"  {markers_path}\n"
        f"Run SpatialBio_Transcriptomics_workflow.ipynb first."
    )

adata_hvg = sc.read_h5ad(adata_path)
cluster_markers = pd.read_csv(markers_path)

print("Loaded AnnData object:", adata_hvg)
print("Cluster column available:", "leiden" in adata_hvg.obs.columns)
print("Cluster markers table:", cluster_markers.shape)
print("Moran's I available:", "moranI" in adata_hvg.uns)


Loaded AnnData object: AnnData object with n_obs × n_vars = 4869 × 2000
    obs: 'in_tissue', 'array_row', 'array_col', 'pxl_row_in_fullres', 'pxl_col_in_fullres', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'n_counts', 'leiden', 'pct_counts_mt'
    var: 'gene_ids', 'feature_types', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
    uns: 'hvg', 'leiden', 'leiden_colors', 'log1p', 'moranI', 'neighbors', 'pca', 'rank_genes_groups', 'spatial', 'spatial_neighbors', 'umap'
    obsm: 'X_pca', 'X_umap', 'spatial'
    varm: 'PCs'
    obsp: 'connectivities', 'distances', 'spatial_connectivities', 'spatial_distances'
Cluster column available: True
Cluster markers table: (2400

In [ ]:
print(list(adata_hvg.var_names[:50]))


['TNFRSF18', 'TNFRSF4', 'CPTP', 'TAS1R3', 'MMP23B', 'TNFRSF14', 'AL139246.3', 'MMEL1', 'PRDM16-DT', 'MEGF6', 'SMIM1', 'KCNAB2', 'TNFRSF25', 'UTS2', 'H6PD', 'PIK3CD', 'AL357140.1', 'AL021155.5', 'TNFRSF1B', 'PDPN', 'EFHD2', 'CTRC', 'HSPB7', 'SPATA21', 'PLA2G2A', 'PLA2G2D', 'ALPL', 'HSPG2', 'WNT4', 'C1QA', 'C1QC', 'C1QB', 'LUZP1', 'ID3', 'RUNX3', 'FAM110D', 'CD52', 'ZNF683', 'MAP3K6', 'FGR', 'AL020997.3', 'PTAFR', 'SDC3', 'TINAGL1', 'COL16A1', 'FAM167B', 'LCK', 'SYNC', 'GJA4', 'MAP7D1']


## Copilot helper functions

Functions to expose common spatial biology questions as reusable analytical tools.


In [ ]:
import os
import re
import numpy as np
import pandas as pd


def _to_dense_vector(x):
    if hasattr(x, "toarray"):
        return np.asarray(x.toarray()).ravel()
    return np.asarray(x).ravel()


def available_clusters(adata, cluster_key="leiden"):
    if cluster_key not in adata.obs:
        raise ValueError(f"Cluster key '{cluster_key}' not found in adata.obs.")
    return sorted(adata.obs[cluster_key].astype(str).unique(), key=lambda x: str(x))


def gene_expression_by_cluster(adata, gene, cluster_key="leiden"):
    gene_lookup = {g.upper(): g for g in adata.var_names}
    gene_upper = gene.upper()

    if gene_upper not in gene_lookup:
        return pd.DataFrame({"error": [f"Gene '{gene}' not found in adata.var_names."]})

    original_gene = gene_lookup[gene_upper]
    values = _to_dense_vector(adata[:, original_gene].X)

    df = pd.DataFrame({
        "cluster": adata.obs[cluster_key].astype(str).values,
        "expression": values
    })

    summary = (
        df.groupby("cluster")["expression"]
        .agg(["mean", "median", "std", "count"])
        .sort_values("mean", ascending=False)
        .reset_index()
    )
    summary.insert(0, "gene", original_gene)
    return summary


def top_markers_for_cluster(cluster_markers, cluster, n=10):
    group_col = "group"
    gene_col = "names"

    if group_col not in cluster_markers.columns or gene_col not in cluster_markers.columns:
        return pd.DataFrame({"error": ["cluster_markers must contain 'group' and 'names' columns."]})

    cluster = str(cluster)
    matched = cluster_markers[cluster_markers[group_col].astype(str) == cluster].copy()

    if matched.empty:
        return pd.DataFrame({"error": [f"Cluster '{cluster}' not found in cluster_markers."]})

    sort_cols = [c for c in ["pvals_adj", "scores"] if c in matched.columns]
    if "pvals_adj" in sort_cols:
        matched = matched.sort_values(["pvals_adj", "scores"], ascending=[True, False])
    elif "scores" in sort_cols:
        matched = matched.sort_values("scores", ascending=False)

    selected_cols = [c for c in ["group", "names", "scores", "logfoldchanges", "pvals", "pvals_adj", "pts"] if c in matched.columns]
    return matched[selected_cols].head(n)


def spatial_summary_by_cluster(adata, cluster_key="leiden"):
    if "spatial" not in adata.obsm:
        return pd.DataFrame({"error": ["No spatial coordinates found in adata.obsm['spatial']."]})

    coords = pd.DataFrame(adata.obsm["spatial"], columns=["x", "y"])
    coords["cluster"] = adata.obs[cluster_key].astype(str).values

    summary = (
        coords.groupby("cluster")
        .agg(
            mean_x=("x", "mean"),
            mean_y=("y", "mean"),
            spread_x=("x", "std"),
            spread_y=("y", "std"),
            n_spots=("cluster", "count")
        )
        .reset_index()
    )
    return summary


def top_spatially_variable_genes(adata, n=10):
    if "moranI" not in adata.uns:
        return pd.DataFrame({"error": ["Moran's I results not found. Run sq.gr.spatial_autocorr first."]})

    moran = adata.uns["moranI"].copy()
    return moran.sort_values("I", ascending=False).head(n)


def summarize_cluster_profile(cluster_markers, cluster, n=5):
    markers = top_markers_for_cluster(cluster_markers, cluster, n=n)
    if "error" in markers.columns:
        return markers.iloc[0]["error"], markers

    genes = ", ".join(markers["names"].astype(str).tolist())
    answer = f"Cluster {cluster} is characterized by top marker genes including {genes}."
    return answer, markers


def find_gene_in_adata(adata, gene_query):
    gene_query = str(gene_query).upper()
    gene_lookup = {str(g).upper(): str(g) for g in adata.var_names}

    if gene_query in gene_lookup:
        return gene_lookup[gene_query]

    partial = [str(g) for g in adata.var_names if gene_query in str(g).upper()]
    if partial:
        return partial[0]

    return None


def extract_gene_from_question(question, gene_names):
    tokens = re.findall(r"[A-Za-z0-9_.-]+", question)

    gene_lookup = {str(g).upper(): str(g) for g in gene_names}

    for token in tokens:
        token_upper = token.upper()
        if token_upper in gene_lookup:
            return gene_lookup[token_upper]

    return None


def extract_cluster_from_question(question, clusters):
    q = question.lower()
    for cluster in clusters:
        patterns = [
            f"cluster {cluster}".lower(),
            f"cluster_{cluster}".lower(),
            f"leiden {cluster}".lower(),
            str(cluster).lower(),
        ]
        if any(p in q for p in patterns):
            return str(cluster)
    return None


def anndata_copilot(question, adata, cluster_markers, cluster_key="leiden"):
    q = question.lower()
    clusters = available_clusters(adata, cluster_key=cluster_key)

    gene = extract_gene_from_question(question, list(adata.var_names))
    cluster = extract_cluster_from_question(question, clusters)

    # Moran/spatial autocorrelation
    if any(term in q for term in ["moran", "autocorrelation", "spatially variable", "spatial variable"]):
        table = top_spatially_variable_genes(adata, n=10)
        if "error" in table.columns:
            return table.iloc[0]["error"], table
        genes = ", ".join(table.index.astype(str).tolist()[:5])
        return f"The top spatially variable genes by Moran's I include: {genes}.", table

    #Immune-related marker enrichment
    if "immune" in q:
        immune_markers = [
            "PTPRC", "CD52", "LCK", "CD3D", "CD3E", "CD8A", "CD4",
            "MS4A1", "CD79A", "CXCL13", "C1QA", "C1QB", "C1QC",
            "LYZ", "HLA-DPA1", "HLA-DPB1", "IL32"
        ]
        rows = []
        for marker in immune_markers:
            if marker in adata.var_names:
                summary = gene_expression_by_cluster(adata, marker, cluster_key)
                top = summary.iloc[0]
                rows.append({
                    "marker": marker,
                    "top_cluster": top["cluster"],
                    "mean_expression": top["mean"]
                })
        table = pd.DataFrame(rows).sort_values("mean_expression", ascending=False)
        answer = "Immune-related marker signals are strongest in clusters: " + \
                 ", ".join(table["top_cluster"].astype(str).unique()[:5])
        return answer, table

    # Gene expression questions
    if gene and any(word in q for word in ["express", "expression", "highest", "enriched", "cluster"]):
        table = gene_expression_by_cluster(adata, gene, cluster_key=cluster_key)
        if "error" in table.columns:
            return table.iloc[0]["error"], table
        top = table.iloc[0]
        return (
            f"{gene} has the highest average expression in Leiden cluster {top['cluster']} "
            f"(mean log-normalized expression = {top['mean']:.3f})."
        ), table

    # 4) Cluster marker / interpretation questions
    if cluster is not None and any(word in q for word in [
        "marker", "markers", "genes", "profile", "characterize",
        "characteristics", "summarize", "interpretation", "interpret"
    ]):
        return summarize_cluster_profile(cluster_markers, cluster, n=8)

    # 5) Spatial coordinate summaries LAST
    if any(word in q for word in ["spatial summary", "location", "coordinates", "where"]):
        table = spatial_summary_by_cluster(adata, cluster_key=cluster_key)
        return "Here is a cluster-level spatial coordinate summary based on the AnnData spatial coordinates.", table

    return (
        "I could not map this question to a supported analysis. Try asking about gene expression, cluster markers, immune enrichment, spatial summaries, or Moran's I.",
        pd.DataFrame()
    )

## Copilot questions

Questions show spatial autocorrelation, immune niche analysis, marker discovery, biological interpretation, and AnnData / Scanpy querying


In [ ]:
example_questions = [
    "Which genes have the highest Moran's I scores?",
    "Which clusters appear enriched for immune-related markers?",
    "What are the top marker genes for cluster 0?",
    "Generate a concise interpretation of cluster 3 markers.",
    "Which Leiden cluster expresses C1QA the most?",
    "Which Leiden cluster expresses CXCL13?",
    "Give me a spatial summary of the Leiden clusters."
]

for question in example_questions:
    print("=" * 100)
    print("Question:", question)
    answer, result_table = anndata_copilot(question, adata_hvg, cluster_markers)
    print("Answer:", answer)
    display(result_table.head(10))


Question: Which genes have the highest Moran's I scores?
Answer: The top spatially variable genes by Moran's I include: IGLC2, IGHG3, CXCL14, IGKC, SCGB1D2.


,I,pval_norm,var_norm,pval_norm_fdr_bh
IGLC2,0.814477,0.0,0.000071,0.0
IGHG3,0.793000,0.0,0.000071,0.0
CXCL14,0.785982,0.0,0.000071,0.0
IGKC,0.773139,0.0,0.000071,0.0
SCGB1D2,0.753445,0.0,0.000071,0.0
SCGB2A2,0.730703,0.0,0.000071,0.0
S100G,0.713002,0.0,0.000071,0.0
CRISP3,0.680897,0.0,0.000071,0.0
IGHG1,0.674273,0.0,0.000071,0.0
IGLC3,0.649576,0.0,0.000071,0.0


Question: Which clusters appear enriched for immune-related markers?
Answer: Immune-related marker signals are strongest in clusters: 0, 10


,marker,top_cluster,mean_expression
1,CD52,0,1.339767
15,HLA-DPB1,0,1.327078
14,HLA-DPA1,0,1.285449
16,IL32,0,1.229961
10,C1QA,0,1.228992
13,LYZ,0,1.200602
11,C1QB,0,1.191820
0,PTPRC,0,1.055360
12,C1QC,0,1.043838
4,CD3E,0,0.921293


Question: What are the top marker genes for cluster 0?
Answer: Cluster 0 is characterized by top marker genes including HLA-DPB1, HLA-DPA1, C1QA, C1QB, C3, LYZ, CD52, IL32.


...

,gene,cluster,mean,median,std,count
0,C1QA,0,1.228992,1.294665,0.751024,755
1,C1QA,10,0.761628,0.922929,0.912163,91
2,C1QA,2,0.476901,0.672821,1.010149,824
3,C1QA,3,0.106785,0.121172,0.624650,327
4,C1QA,1,0.083255,0.136280,0.646599,72
5,C1QA,6,-0.245202,-0.202978,0.665179,377
6,C1QA,4,-0.312511,-0.317910,0.641443,237
7,C1QA,7,-0.463014,-0.432845,0.607328,428
8,C1QA,9,-0.502202,-0.469723,0.680801,498
9,C1QA,8,-0.574155,-0.549644,0.630783,783


Question: Which Leiden cluster expresses CXCL13?
Answer: CXCL13 has the highest average expression in Leiden cluster 0 (mean log-normalized expression = 0.479).


,gene,cluster,mean,median,std,count
0,CXCL13,0,0.479019,-0.233836,1.757080,755
1,CXCL13,6,0.136363,-0.233836,1.196553,377
2,CXCL13,2,-0.015353,-0.233836,1.047130,824
3,CXCL13,1,-0.087614,-0.233836,0.619044,72
4,CXCL13,5,-0.096754,-0.233836,0.564262,374
5,CXCL13,3,-0.106749,-0.233836,0.614632,327
6,CXCL13,4,-0.118030,-0.233836,0.652705,237
7,CXCL13,9,-0.125603,-0.233836,0.573973,498
8,CXCL13,7,-0.137810,-0.233836,0.545448,428
9,CXCL13,8,-0.171264,-0.233836,0.420503,783


Question: Give me a spatial summary of the Leiden clusters.
Answer: Here is a cluster-level spatial coordinate summary based on the AnnData spatial coordinates.


,cluster,mean_x,mean_y,spread_x,spread_y,n_spots
0,0,21725.488742,12015.883444,4445.614466,4626.281738,755
1,1,23547.000000,16100.722222,1038.735634,723.948450,72
2,10,23349.692308,11431.989011,1733.982319,2655.444246,91
3,11,24661.495146,21650.941748,4843.783801,2250.310383,103
4,2,23188.781553,10306.304612,4337.763219,3706.009382,824
5,3,20955.082569,17986.990826,3900.784435,2511.625792,327
6,4,25564.527426,14701.687764,2150.299361,1247.138006,237
7,5,21259.120321,9706.858289,6330.439394,4534.988595,374
8,6,21101.400531,9841.010610,5990.852095,2081.282944,377
9,7,17691.670561,17371.385514,4645.020809,2336.665453,428


##LLM-assisted interpretation

Sends only the validated computed answer and table to an LLM for concise interpretation. The LLM is grounded in the computed table and instructed not to invent claims.



In [ ]:
# Set OpenAI API key safely.
# The LLM assisted section here will skip LLM interpretation if no key is available.

import os
os.environ["OPENAI_API_KEY"] = "sk-..."   #<<<<<< --------- Need valid key

In [ ]:
def llm_interpretation(question, answer, result_table, model="gpt-4o-mini"):
    """LLM interpretation grounded strictly in computed AnnData results."""
    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_key:
        return "OPENAI_API_KEY is not set. Skipping LLM interpretation."

    try:
        from openai import OpenAI
        client = OpenAI(api_key=api_key)

        # Keep only visible, computed evidence
        table_preview = result_table.head(12).copy()
        table_text = table_preview.to_csv(index=True)

        prompt = f"""
You are assisting with exploratory spatial transcriptomics and computational pathology analysis.

STRICT RULES:
- Use ONLY the computed answer and computed table below.
- Do NOT mention genes, clusters, pathways, cell types, or markers that are not present in the computed table.
- Do NOT invent unsupported biological claims.
- If the evidence is suggestive but not definitive, clearly say so.
- Prefer cautious language such as "suggests", "is consistent with", or "may indicate".
- Do not diagnose disease or make clinical claims.

Question:
{question}

Computed answer:
{answer}

Computed table:
{table_text}

Write the response in this format:

1. Concise interpretation:
Write 4-6 careful sentences grounded only in the genes and values shown above.

2. Supporting evidence:
List 3-6 specific genes or computed values from the table that support the interpretation.

3. Recommended follow-up:
Suggest one visualization, validation, or computational follow-up.

4. Caution:
State one limitation or reason this result should not be over-interpreted.
"""

        response = client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are a careful biomedical data analysis assistant. "
                        "You must stay strictly grounded in the computed table. "
                        "Do not infer genes or markers that are not shown."
                    )
                },
                {"role": "user", "content": prompt}
            ],
            temperature=0.1
        )

        return response.choices[0].message.content

    except Exception as e:
        return f"LLM interpretation failed: {e}"





## Question

In [ ]:
question = "Generate a concise interpretation of cluster 3 markers."


## Answer

In [ ]:
answer, result_table = anndata_copilot(question, adata_hvg, cluster_markers)

print("Question:", question)
print("Computed answer:", answer)
display(result_table.head(10))

print("\nLLM-assisted interpretation:\n")
print(llm_interpretation(question, answer, result_table))

Question: Generate a concise interpretation of cluster 3 markers.
Computed answer: Cluster 3 is characterized by top marker genes including CXCL14, MMP11, IGHE, TGM2, COL4A1, TTLL12, COL18A1, CRAT.


,group,names,scores,logfoldchanges,pvals,pvals_adj
6000,3,CXCL14,19.797651,NaN,3.119010e-87,6.238021e-84
6001,3,MMP11,19.576500,NaN,2.453056e-85,2.453056e-82
6002,3,IGHE,18.978197,NaN,2.583054e-80,1.722036e-77
6003,3,TGM2,17.872606,NaN,1.927725e-71,9.638625e-69
6004,3,COL4A1,17.214592,NaN,2.063977e-66,8.255910e-64
6005,3,TTLL12,17.171540,NaN,4.337663e-66,1.445888e-63
6006,3,COL18A1,16.767464,NaN,4.221483e-63,1.206138e-60
6007,3,CRAT,16.732292,NaN,7.624678e-63,1.906170e-60



LLM-assisted interpretation:

1. Concise interpretation:  
Cluster 3 is characterized by a set of marker genes that exhibit high expression levels, suggesting a potentially distinct functional role within the analyzed tissue. The markers include CXCL14, MMP11, and IGHE, which have exceptionally low p-values, indicating strong statistical significance. The presence of extracellular matrix-related genes such as COL4A1 and COL18A1 may suggest involvement in structural or remodeling processes. Additionally, TGM2 and CRAT could imply metabolic or signaling functions. Overall, the expression profile of these markers may indicate a specialized biological context for Cluster 3.

2. Supporting evidence:  
- CXCL14 has a score of 19.797651 and a p-value of 3.119010371949939e-87.  
- MMP11 has a score of 19.5765 and a p-value of 2.453055937498904e-85.  
- IGHE has a score of 18.978197 and a p-value of 2.5830540755315648e-80.  
- COL4A1 has a score of 17.214592 and a p-value of 2.063977390557825e